In [ ]:
# scraping the sections from the https://devgan.in/bns/section/{}/
import json
import requests
from bs4 import BeautifulSoup
from concurrent.futures import ThreadPoolExecutor

def scrape_single_url(url, section_type="BNS"):
    """
    Scrape a single BNS section
    
    Args:
        url: URL to scrape
        section_type: Law code type (default: "BNS")
    
    Returns:
        Dictionary with scraped data or None if error
    """
    try:
        section = url.split('/')[-2]
        print(f"Scraping section {section}...")

        response = requests.get(url)
        soup = BeautifulSoup(response.text, 'html.parser')

        p_elements = [p.get_text().strip() for p in soup.find_all('p') if p.get_text().strip()]
        h2_elements = [h2.get_text().strip() for h2 in soup.find_all('h2') if h2.get_text().strip()]
        h3_elements = [h3.get_text().strip() for h3 in soup.find_all('h3') if h3.get_text().strip()]
        li_elements = [li.get_text().strip() for li in soup.find_all('li') if li.get_text().strip()]

        combined_content = '\n'.join(h2_elements + h3_elements + p_elements + li_elements)

        return {
            'law_code': section_type,
            'section': section,
            'content': combined_content,
            'source_url': url
        }
    except Exception as e:
        print(f"Error scraping section {section}: {str(e)}")
        return None

# Generate URLs
base_url = "https://devgan.in/bns/section/{}/"
urls = [base_url.format(i) for i in range(1, 359)]

# Scrape with 10 threads at once (fast but not overwhelming the server)
print(f"Starting to scrape {len(urls)} sections...\n")

with ThreadPoolExecutor(max_workers=10) as executor:
    results = list(executor.map(scrape_single_url, urls))

# Remove None values (errors)
scraped_data = [d for d in results if d is not None]

print(f"\nSuccessfully scraped {len(scraped_data)} sections")

# Save raw data
import os
os.makedirs("../data/raw", exist_ok=True)

with open("../data/raw/bns_sections.json", "w", encoding="utf-8") as f:
    json.dump(scraped_data, f, indent=2, ensure_ascii=False)

print(f"Saved to ../data/raw/bns_sections.json")

Scraping section 1 (1/358)...
Scraping section 2 (2/358)...
Scraping section 3 (3/358)...
Scraping section 4 (4/358)...
Scraping section 5 (5/358)...
Scraping section 6 (6/358)...
Scraping section 7 (7/358)...
Scraping section 8 (8/358)...
Scraping section 9 (9/358)...
Scraping section 10 (10/358)...
Scraping section 11 (11/358)...
Scraping section 12 (12/358)...
Scraping section 13 (13/358)...
Scraping section 14 (14/358)...
Scraping section 15 (15/358)...
Scraping section 16 (16/358)...
Scraping section 17 (17/358)...
Scraping section 18 (18/358)...
Scraping section 19 (19/358)...
Scraping section 20 (20/358)...
Scraping section 21 (21/358)...
Scraping section 22 (22/358)...
Scraping section 23 (23/358)...
Scraping section 24 (24/358)...
Scraping section 25 (25/358)...
Scraping section 26 (26/358)...
Scraping section 27 (27/358)...
Scraping section 28 (28/358)...
Scraping section 29 (29/358)...
Scraping section 30 (30/358)...
Scraping section 31 (31/358)...
Scraping section 32 (32/35

In [ ]:
# load and clean scraped data
import json
import re

# Load scraped data
with open("../data/raw/bns_sections.json", "r", encoding="utf-8") as f:
    data = json.load(f)

print(f"Loaded {len(data)} sections")

# Define cleaning patterns
pattern_nav = re.compile(r'\b(?:Home|Top|Back|Prev|Index|Next)\b', re.IGNORECASE)

def clean_text(text):
    text = text.replace("\n", " ").replace("\t", " ")
    text = pattern_nav.sub(" ", text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

# Clean all sections
for item in data:
    item["content"] = clean_text(item["content"])

# Save cleaned data
import os
os.makedirs("../data/processed", exist_ok=True)

with open("../data/processed/cleaned_final.json", "w", encoding="utf-8") as f:
    json.dump(data, f, indent=2, ensure_ascii=False)

print(f"Cleaned and saved {len(data)} sections to ../data/processed/cleaned_final.json")

In [ ]:
# chunking
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=800)

# Create a Langchain document for each JSON record 
documents = []
for record in data:
    document = Document(
        page_content=record['content'],
        metadata={'section':record['section']}
    )
    documents.append(document)

docs = splitter.split_documents(documents)    

Bharatiya Nyaya Sanhita S. 1 Short title, commencement and application. Description Preamble. An Act to consolidate and amend the provisions relating to offences and for matters connected therewith or incidental thereto. Be it enacted by Parliament in the Seventy-fourth Year of the Republic of India as follows: This Act may be called the Bharatiya Nyaya Sanhita, 2023. It shall come into force on such date as the Central Government may, by notification in the Official Gazette, appoint, and different dates may be appointed for different provisions of the Sanhita. Every person shall be liable to punishment under this Sanhita and not otherwise for every act or omission contrary to the provisions thereof, of which he shall be guilty within India. Any person liable, by any law for the time being in force in India, to be tried for an offence committed beyond India shall be dealt with according to the provisions of this Sanhita for any act committed beyond India in the same manner as if such a

In [105]:
print(len(docs))

538


In [142]:
docs[1]

Document(metadata={'section': '1'}, page_content='registered in India wherever it may be; any person in any place without and beyond India committing offence targeting a computer resource located in India. Explanation: In this section the word “offence” includes every act committed outside India which, if committed in India, would be punishable under this Sanhita. Illustration: A, who is a citizen of India, commits a murder in any place without and beyond India, he can be tried and convicted of murder in any place in India in which he may be found. any citizen of India in any place without and beyond India; any person on any ship or aircraft registered in India wherever it may be; any person in any place without and beyond India committing offence targeting a computer resource located in India. Explanation: In this section the word “offence” includes every act committed outside India which, if committed in India, would be punishable under this Sanhita. Illustration: A, who is a citizen

In [128]:
from dotenv import load_dotenv
import os 

load_dotenv()

True

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

model_name = "BAAI/bge-base-en-v1.5"
model_kwargs={'device': 'cpu'}
encode_kwargs={'normalize_embeddings' : 'False'}
embedding = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4860.26it/s]
BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [159]:
# embedding and vector store 

from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient

# connect to Qdrant Cloud
client = QdrantClient(
    url=os.getenv("QDRANT_CLOUD_URL"),
    api_key=os.getenv("QDRANT_CLOUD_API_KEY"),
    prefer_grpc=True
)

vectorstore = QdrantVectorStore(
    client=client,
    embedding=embedding,
    collection_name=os.getenv('COLLECTION_NAME')
)

In [160]:
# verifying if collection exists

from qdrant_client import models

is_exist = client.collection_exists(collection_name=os.getenv('COLLECTION_NAME'))

print('Collection already exists' if is_exist else 'Creating Collection : ', os.getenv('COLLECTION_NAME'))

if is_exist == False:
    client.create_collection(
    collection_name=os.getenv('COLLECTION_NAME'),
    vectors_config=models.VectorParams(size=768, distance=models.Distance.COSINE),
    )

Collection already exists bns-sections


In [161]:
docs[0]

Document(metadata={'section': '1'}, page_content='Bharatiya Nyaya Sanhita S. 1 Short title, commencement and application. Description Preamble. An Act to consolidate and amend the provisions relating to offences and for matters connected therewith or incidental thereto. Be it enacted by Parliament in the Seventy-fourth Year of the Republic of India as follows: This Act may be called the Bharatiya Nyaya Sanhita, 2023. It shall come into force on such date as the Central Government may, by notification in the Official Gazette, appoint, and different dates may be appointed for different provisions of the Sanhita. Every person shall be liable to punishment under this Sanhita and not otherwise for every act or omission contrary to the provisions thereof, of which he shall be guilty within India. Any person liable, by any law for the time being in force in India, to be tried for an offence committed beyond India shall be dealt with according to the provisions of this Sanhita for any act comm

In [162]:
# saving data in qdrant
vectorstore.add_documents(docs)

['a7a090bf96204a729f043a51d4f4c8c3',
 '9e15c4b9fad644f680a5a18e660f00c4',
 'c0d5d637cc07412189789199d95e14bf',
 'e118ff32748b42f1aa32e9efd9204dad',
 'e642589de8044dadbe679ecc4b91463a',
 '77c7929fdf704cdcab9cce5793c94dc1',
 '412bd7b894d54e149d7a16892ce05004',
 '52c5c4acdc5742b698288682c24989c6',
 'eecd8c001eba4601b2e6200750fcda20',
 'e03bc592be334d18a685bb205b77a424',
 'ca7235f3f5a64b0ba078714fb3692aeb',
 '24b476c177bc4c11baca132de86b90cb',
 'b50bc90f14b94eba83cfcc35cf28b641',
 '4b46b20be0244ca3903d0b319861ca04',
 '57f95f08292d4fc69664df0465da90a8',
 '8ed8dbcfa697426ba9ea94e35ceac342',
 '790734ffd9864a7fa10b040aa4e831d5',
 '90a08441c18744dba7b235c280b7059f',
 '1d3cdf6034ce4ff788579f14811da7fc',
 '8b09c5e79de942e1a800c8a0e1267ee4',
 '85268a3a61934dd0980dd9fc46bb10a2',
 'fe4f08326fd24ca9a3ee03e4e1d86255',
 '5fbf976c3d6045c2b99d8b03b814d98d',
 'f49e04a71aa54949aeb9ff0107d6c501',
 'dcdd1e96089549ff9d75779c1d53015b',
 '55df75bf8c1148868e6d67e38f9bb29f',
 '098a0d0d38d84bc0a33a9eec7b59fe5f',
 